# Generic Scattering Calculator: I(q) from a real-space structure

`GenericScatteringCalculator` is the nuclear, 1D part of SasView's *Generic
Scattering Calculator*. It reads a structure — atoms from a **PDB** file, pixels
from an **SLD** file, or the cells of a **VTK** mesh — and computes the
orientationally averaged scattering with the exact Debye formula:

$$
w_j = (\rho_j - \rho_\mathrm{solvent})\,v_j, \qquad
I(q) = \frac{\mathrm{scale}\cdot 10^8}{V_\mathrm{norm}}
\sum_{j,k} w_j w_k \frac{\sin q r_{jk}}{q r_{jk}} + \mathrm{background}
$$

This notebook:

1. builds a sphere of pixels in a box of D₂O and saves it as an SLD file;
2. loads it, draws its points and computes I(q);
3. checks the curve against sasmodels' analytic sphere;
4. shows what the normalisation volume does near contrast match;
5. fits the computed curve with `SANSFitter`;
6. computes a PDB structure (C60) in vacuum, and shows why PDB curves in a
   solvent are only qualitative;
7. loads a VTK mesh.

In [ ]:
import os
import tempfile
import warnings

import numpy as np
import plotly.graph_objects as go
from sasmodels.core import load_model
from sasmodels.direct_model import call_kernel

from sans_fitter import GenericScatteringCalculator, SANSFitter

DATA = os.path.join('..', 'examples', 'data')
RADIUS, STEP = 30.0, 3.0  # Å
SLD_PARTICLE, SLD_D2O = 1.0, 6.34  # 10⁻⁶ Å⁻², the unit of every sasmodels SLD

## 1. A sphere of pixels as an SLD file

An SLD file has one header line, then `x y z sld` per pixel, with positions in
Å and the SLD in Å⁻² (the file convention, which SasView also uses). Pixels
outside the sphere carry the solvent's SLD. At that solvent they do not
scatter, and they do not count towards the normalisation volume.

In [ ]:
n = int(RADIUS / STEP) + 1
axis = np.arange(-n, n + 1) * STEP
xyz = np.stack(np.meshgrid(axis, axis, axis, indexing='ij'), -1).reshape(-1, 3)
inside = (xyz**2).sum(axis=1) <= RADIUS**2
sld = np.where(inside, SLD_PARTICLE, SLD_D2O) * 1e-6

path = os.path.join(tempfile.mkdtemp(), 'sphere.sld')
np.savetxt(path, np.column_stack([xyz, sld]), header='x y z sld', comments='')
print(f'{len(xyz)} pixels, {inside.sum()} inside the sphere')

## 2. Load, look, compute

The load line reports the lattice step and the pixel volume it inferred.
`plot_points()` is SasView's *Draw Points*: pixels are coloured by SLD, and
the ones that match the solvent are drawn translucent yellow.

In [ ]:
gsc = GenericScatteringCalculator()
gsc.load_structure(path)
gsc.set_params(sld_solvent=SLD_D2O)  # 10⁻⁶ Å⁻², like fit.constrain('d2o.sld_solvent', 6.34)
gsc.plot_points()

In [ ]:
gsc.describe()

Pixels on a lattice use an FFT autocorrelation of the weight grid. It gives
the same Debye sum exactly, in milliseconds. Atoms and mesh cells use the
pair sum, which grows as the square of the number of points.

In [ ]:
profile = gsc.calculate()
profile

In [ ]:
gsc.plot_profile()

## 3. Check against the analytic sphere

Because the normalisation volume is the volume of the points that scatter,
the pixel sphere at `scale=1` should match sasmodels' `sphere` at `scale=1`.
The difference left is the staircase surface of the pixels.

In [ ]:
kernel = load_model('sphere').make_kernel([profile.q])
analytic = call_kernel(
    kernel,
    {'radius': RADIUS, 'sld': SLD_PARTICLE, 'sld_solvent': SLD_D2O, 'scale': 1, 'background': 0},
)
low_q = profile.q * RADIUS <= 3
deviation = profile.intensity[low_q] / analytic[low_q] - 1
print(f'qR <= 3: {deviation.min():+.2%} to {deviation.max():+.2%}')

fig = profile.plot(show=False)
fig.add_trace(
    go.Scatter(x=profile.q, y=analytic, mode='lines', name='sasmodels sphere', line={'dash': 'dash'})
)
fig

## 4. The normalisation volume near contrast match

When the solvent exactly matches a population of points, those points stop
counting towards `V_norm`, and the curve jumps by the volume ratio. Here the
box pixels match D₂O exactly. Moving the solvent by a tiny amount makes them
count again: the Debye sum barely changes, but the curve drops by
`V_all / V_active`.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter('ignore')  # the near-match warning is the point here
    gsc.set_params(sld_solvent=SLD_D2O + 1e-9)
    shifted = gsc.calculate()
print(
    f'V_norm {profile.normalisation_volume:.0f} -> {shifted.normalisation_volume:.0f} Å³, '
    f'I(q_min) ratio {profile.intensity[0] / shifted.intensity[0]:.2f}'
)

For a contrast-variation series, state the volume and the jump disappears:

In [ ]:
gsc.set_params(normalisation_volume=profile.normalisation_volume)
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    stated = gsc.calculate()
print(f'I(q_min) ratio with a stated volume: {profile.intensity[0] / stated.intensity[0]:.4f}')
gsc.set_params(sld_solvent=SLD_D2O, normalisation_volume='auto')

## 5. Fit the computed curve

The profile is theory: `to_data1d()` gives a `Data1D` with no dI and no dQ.
State the weighting and the resolution explicitly, then fit as usual.

In [ ]:
profile = gsc.calculate()
fitter = SANSFitter()
fitter.set_data(profile.to_data1d())
fitter.set_weighting('abs')  # σ = |I|: equal relative weight across decades
fitter.set_resolution('none')
fitter.set_model('sphere')
fitter.set_param('sld', value=SLD_PARTICLE, vary=False)
fitter.set_param('sld_solvent', value=SLD_D2O, vary=False)
fitter.set_param('background', value=0.0, vary=False)
fitter.set_param('radius', value=25, min=10, max=60, vary=True)
fitter.set_param('scale', value=1.0, min=0.5, max=2, vary=True)
result = fitter.fit(method='lm')

## 6. A PDB structure: C60

Atoms are coloured by element, and `CONECT` records are drawn as bonds (this
file only connects one pentagon). The load line reports Rg.

In [ ]:
c60 = GenericScatteringCalculator(os.path.join(DATA, 'c60.pdb'))
c60.set_q_grid(1e-2, 2.0, 100)
c60.plot_points()

In [ ]:
c60.calculate()
c60.plot_profile()

### Why PDB curves in a solvent are only qualitative

Each atom's volume is SasView's *natural atomic volume*, from the element's
bulk density: liquid H₂, N₂ and O₂ for H, N and O. A typical protein residue
then gets about 310 Å³ instead of about 133 Å³, and in D₂O a protein comes
out about 8× too intense at q → 0. `calculate()` warns, with the ratio of the
elemental volumes to a protein-like estimate for this structure. C60 has no
hydrogen, so here the ratio is below 1; the comparison only means something
for proteins, but the volumes are wrong either way.

In [ ]:
c60.set_params(sld_solvent=SLD_D2O)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    c60.calculate()
for warning in caught:
    print(warning.message)

## 7. A VTK mesh

VTK cells become points at their volume centroids, carrying the cell volume
and SLD. This mesh is SasView's cube of five tetrahedra: one central cell of
volume 8/3 and four corner cells of 4/3 each.

In [ ]:
cube = GenericScatteringCalculator(os.path.join(DATA, 'five_tetrahedra_cube.vtk'))
print(cube.structure.volume, cube.structure.volume.sum())
cube.plot_points()

## Summary

- `load_structure()` reads PDB, SLD and VTK files; `plot_points()` draws them.
- `sld_solvent` is in 10⁻⁶ Å⁻², like every sasmodels SLD.
- `calculate()` returns a `ScatteringProfile`. Lattices use an exact FFT
  path; atoms and cells use the exact pair sum.
- `V_norm` is the volume of the points that scatter, so `scale` is a volume
  fraction for pixels and cells. State it explicitly for contrast series.
- `profile.to_data1d()` feeds `SANSFitter`; state the weighting and resolution.
- PDB curves in a solvent are only qualitative.